# 🔍 EDA & Baseline Analysis

**Цель:** понять данные, выявить паттерны, сформулировать гипотезы фич

**Автор:** Антон Лысогор

**Дата:** 2026-05-01

## Импорт библиотек и загрузка данных

In [1]:
# 📦 Импорт библиотек
import os
import sys
from pathlib import Path

import pandas as pd

In [2]:
# Проверка, что используется именно окружение проекта
# Путь должен быть внури проекта ~/ml-hackathon/.venv/bin/python
print(f"Путь к интерпретатору: {sys.executable}")

Путь к интерпретатору: /home/itj/ml-hackathon/.venv/bin/python


In [3]:
# 🗄️ Загрузка данных

#  Адаптация путей под проект
#
# Всегда вычисляем корень относительно ПАПКИ ноутбука,
# а не относительно того, где мы сейчас «гуляем» через chdir
# Если ноутбук в notebooks/eda, то корень — это два уровня вверх
ROOT = Path(os.getcwd())

# Проверка: если мы уже в корне или ушли не туда,
# этот блок поможет не подняться выше проекта (ml-hackathon)
if "notebooks" in str(ROOT):
    ROOT = ROOT.parent.parent

DATA_ROOT = ROOT / "data" / "train"

print(f"Рабочий корень ROOT: {ROOT}")
print(f"Файлы будут искаться в DATA_ROOT: {DATA_ROOT}")

# DATA_ROOT = Path('data/train')
user = pd.read_csv(DATA_ROOT / "user.csv")
shift = pd.read_csv(DATA_ROOT / "shift.csv")
event = pd.read_csv(DATA_ROOT / "event.csv")

print(f"✅ Loaded: users={len(user)}, shifts={len(shift)}, events={len(event)}")

Рабочий корень ROOT: /home/itj/ml-hackathon
Файлы будут искаться в DATA_ROOT: /home/itj/ml-hackathon/data/train
✅ Loaded: users=5154, shifts=39999, events=383883


## Первичный обзор данных

In [4]:
user

,location_id,is_strict_location,id,has_mk
0,64,False,0a66722f9259c08a6a6d78d2ea90e375,True
1,61,False,314a2bc9a488ba22a31ad0d9a0c1c26f,True
2,181,False,73f5bbba685e5e676f59a4709f04c825,True
3,234,False,50dd1bf4e4ba7ebec4a45076be1dc90c,True
4,158,False,c53805ed7cbcabe72f70da8404213552,True
...,...,...,...,...
5149,152,False,e21e8b9cf5d72eb36e8647bc69df4e07,True
5150,234,False,90d0f5a1b08d738f7e07db432a48d399,True
5151,308,False,6f3459a69e6a17d13aea04f03c988018,True
5152,234,False,87da8e1a8cc41b4a8ee7e39b1e115fd0,True


In [5]:
user.info()

<class 'pandas.DataFrame'>
RangeIndex: 5154 entries, 0 to 5153
Data columns (total 4 columns):
 #   Column              Non-Null Count  Dtype
---  ------              --------------  -----
 0   location_id         5154 non-null   int64
 1   is_strict_location  5154 non-null   bool 
 2   id                  5154 non-null   str  
 3   has_mk              5154 non-null   bool 
dtypes: bool(2), int64(1), str(1)
memory usage: 90.7 KB


In [6]:
df = user
summary = pd.DataFrame({"dtype": df.dtypes, "unique_count": df.nunique()})
print(summary)

                    dtype  unique_count
location_id         int64           210
is_strict_location   bool             1
id                    str          5154
has_mk               bool             2


Таблица ***user*** — всё нормально, пропущенных нет, поле *id* — все значения уникальны, числовые и булевые значения в соответствующих форматах.

In [7]:
shift

,id,start_at,location_id,task_type,employer_id,workplace_id,need_mk,id_differential,hours,reward,capacity
0,9857,2026-01-06 12:30:00+00:00,161,Погрузка и разгрузка товара,c1714b6178cdbe470384bdd7ea181f3c,27b794bbdb945cc982c40394dc9b520a,False,False,3,900.0,2
1,21761,2026-01-15 12:30:00+00:00,161,Погрузка и разгрузка товара,c1714b6178cdbe470384bdd7ea181f3c,27b794bbdb945cc982c40394dc9b520a,False,False,3,900.0,2
2,21647,2026-01-24 12:30:00+00:00,161,Погрузка и разгрузка товара,c1714b6178cdbe470384bdd7ea181f3c,27b794bbdb945cc982c40394dc9b520a,False,False,3,900.0,2
3,38993,2026-01-06 09:00:00+00:00,13,Помощь в торговом зале,e0480c3c5525389f776aa4fa58a7f72b,b74ac875bc93d4c4ae24893c99147640,True,False,7,1800.0,1
4,48240,2026-01-04 16:00:00+00:00,13,Помощь в торговом зале,e0480c3c5525389f776aa4fa58a7f72b,b74ac875bc93d4c4ae24893c99147640,True,False,7,1800.0,1
...,...,...,...,...,...,...,...,...,...,...,...
39994,13998,2026-01-23 21:00:00+00:00,234,Приготовление пищи,02d2b372ccc7404c03bf705d67360723,67cffef2e08cf63c214fc0b9f6273cd5,True,False,12,4300.0,1
39995,56816,2026-02-04 08:00:00+00:00,126,Помощь в прикассовой зоне,d10547569bf9ec2f4879eb99060fe4b5,59f7bf7b6a9364a192992b883daad302,True,False,13,2300.0,2
39996,8585,2026-01-27 09:00:00+00:00,234,Выкладка товара,02d2b372ccc7404c03bf705d67360723,7e17fba7df94be7f98948340eee1935c,True,False,12,3700.0,1
39997,44958,2026-01-30 08:00:00+00:00,181,Выкладка товара,faff2687685943a75600a311b3ef706c,0a892d9071a08af5fc1ac709986289ab,True,False,9,2500.0,1


In [8]:
shift.info()

<class 'pandas.DataFrame'>
RangeIndex: 39999 entries, 0 to 39998
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   id               39999 non-null  int64  
 1   start_at         39999 non-null  str    
 2   location_id      39999 non-null  int64  
 3   task_type        39999 non-null  str    
 4   employer_id      39999 non-null  str    
 5   workplace_id     39999 non-null  str    
 6   need_mk          39999 non-null  bool   
 7   id_differential  39999 non-null  bool   
 8   hours            39999 non-null  int64  
 9   reward           39999 non-null  float64
 10  capacity         39999 non-null  int64  
dtypes: bool(2), float64(1), int64(4), str(4)
memory usage: 2.8 MB


In [9]:
df = shift
summary = pd.DataFrame({"dtype": df.dtypes, "unique_count": df.nunique()})
print(summary)

                   dtype  unique_count
id                 int64         39999
start_at             str          1526
location_id        int64           212
task_type            str            27
employer_id          str            50
workplace_id         str          1638
need_mk             bool             2
id_differential     bool             2
hours              int64            19
reward           float64            76
capacity           int64            14


In [10]:
df.describe()

,id,location_id,hours,reward,capacity
count,39999.000000,39999.000000,39999.000000,39999.000000,39999.000000
mean,31753.630966,167.423286,9.878597,2991.381660,1.189555
std,18398.316334,81.659397,3.036128,1181.450817,1.045574
min,4.000000,1.000000,1.000000,125.000000,1.000000
25%,15776.500000,119.000000,7.000000,2100.000000,1.000000
50%,31751.000000,166.000000,12.000000,3100.000000,1.000000
75%,47687.500000,234.000000,12.000000,3700.000000,1.000000
max,63617.000000,313.000000,22.000000,9500.000000,154.000000


Таблица ***shift***, или *рабочая смена*. Пропущенных значений нет, все id уникальны.

* *start_at* - начало смены,  время в строковой переменной, надо переводить во время или убедиться, что оно в дальнейшем переводится в него.
* 212 локаций, 27 типов работ, 50 заказчиков.
* *workplace_id*: Идентификатор конкретного рабочего места внутри локации. **Примечение:** пока непонятно, что значит.
* *need_mk* — начличие медицинской книжки, связано с полем *has_mk* в таблице *user*.
* *id_differential* — машинный перевод: "Вероятно, признак того, отличается ли оплата в этой смене от стандартной (например, повышенная ставка)." **Примечение:** пока непонятно, что значит.
* *hours* — длительность смены, от 1 часа до 22, средняя 10 часов с отклонением в 3 часа, медианное значение 12 часов.
* *reward* — вознаграждение за смену от 125 до 9500. 125 - низкое значение, стоит проверить. Середина и среднее в районе 3000. Из этого поля можем получить расчётную ставку за один час. Чем она больше, тем больше пользователь будет мотивирован выйти на смену.
* *capacity* — количество человек на смену. Обычно 1, максимум 154, похоже на выброс.

In [11]:
event

,id,shift_id,user_id,interaction,ts
0,ab450739fc9176a3ba7e92e6bbc9b169,12396,69359158a2c25806c20b11f860fa46ab,VIEW,2026-01-06
1,46fd04be38ec4738278ec9e941463802,12396,258962a7b6a7e68cadaaf0a927eadc90,VIEW,2026-01-05
2,8ca0102b2784c383dc430f795b6f5377,12396,258962a7b6a7e68cadaaf0a927eadc90,VIEW,2026-01-03
3,e3004221404186d95b086dc0cb688905,12396,258962a7b6a7e68cadaaf0a927eadc90,VIEW,2026-01-07
4,e5be648f4c7d19f33cbc06409f8525f5,12396,984483ffe4b8fd56a7d9d78e9fe97627,VIEW,2026-01-07
...,...,...,...,...,...
383878,7c66fd41f448e2853bb986a805e1bbbe,46323,d13ec6d546a7cbc0ea091df042489fa0,FINISHED,2026-01-28
383879,64c83f6f916ebcac643b2a05d3015494,46323,af40af56f60b3fbc8e5aadb4fc687f11,SYSTEM_CANCEL,2026-01-25
383880,274ce9d6472799c42697f358af7e55dc,32605,67b8d8989c0a30cef2d33b67b837208e,FINISHED,2026-01-30
383881,246eb0d98729ad0beac5c74c2fd0c27c,53015,c036e3971821db4ae537f9c3a8ddc2cb,SYSTEM_CANCEL,2026-01-28


In [12]:
event.info()

<class 'pandas.DataFrame'>
RangeIndex: 383883 entries, 0 to 383882
Data columns (total 5 columns):
 #   Column       Non-Null Count   Dtype
---  ------       --------------   -----
 0   id           383883 non-null  str  
 1   shift_id     383883 non-null  int64
 2   user_id      383883 non-null  str  
 3   interaction  383883 non-null  str  
 4   ts           383883 non-null  str  
dtypes: int64(1), str(4)
memory usage: 14.6 MB


In [13]:
df = event
summary = pd.DataFrame({"dtype": df.dtypes, "unique_count": df.nunique()})
print(summary)

             dtype  unique_count
id             str        383883
shift_id     int64         30114
user_id        str          3749
interaction    str             5
ts             str            72


In [14]:
df.interaction.value_counts()

interaction
VIEW             318747
APPLY             24030
FINISHED          17305
SYSTEM_CANCEL     12998
USER_CANCEL       10803
Name: count, dtype: int64

In [15]:
df.ts.value_counts()

ts
2026-01-04    19917
2026-01-03    19782
2026-01-02    19191
2026-01-05    19164
2026-01-07    18542
              ...  
2025-12-23        6
2025-12-18        4
2025-12-16        1
2025-12-21        1
2025-12-17        1
Name: count, Length: 72, dtype: int64

Таблица ***event***, пожалуй, самая важная. по видимому представляет лог действий. Пользователь, посмотрел VIEW, при этом может смотреть несколько раз; записался  APPLY и либо вышел FINISHED, либо отказался USER_CANCEL. Ещё есть вариант SYSTEM_CANCEL — смена была отменена.

Пропущенных значений нет.

* *shift_id* — 30k, хотя в таблице *shift* их 39k.
* *user_id* — 3k, а в таблице *user* их 5k. Похоже, что не все смены и все пользователи были активны.
* *ts*, time stamp - в строковом виде.

## Вывод

В целом все данные корректны, только дата тербует преобразования. 

**Дальнейшие шаги:** перед преобразованием и наращиванием фич, теперь надо изучить baseline, что конкретно там уже сделано.